# Security Master Resolution Analysis

## Objective

This notebook investigates the Security Master resolution results before any additional rule is promoted into production code.

The analysis focuses on:

- `RESOLVED` securities, with emphasis on the resolution rules used;
- `AMBIGUOUS` securities for which multiple distinct FIGIs remain;
- `UNRESOLVED` securities for which no usable FIGI survived;
- the relationship between benchmark ticker/exchange attributes and raw OpenFIGI mappings;
- whether `security_type`, `security_type_2`, `ticker`, `name`, `composite_figi`, or `share_class_figi` provide evidence for safe additional deterministic rules.

**Important:** this notebook is diagnostic only. It does not modify production datasets and does not automatically select a FIGI for ambiguous securities.

## 1. Imports and project paths

The notebook expects to run from either the project root or the `notebooks/` directory. Paths are resolved defensively from the current working directory.

In [1]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()

if (cwd / "src").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Unable to locate project root containing src/.")

CANDIDATES_PATH = PROJECT_ROOT / "data/02_processed/securities/security_candidates.csv"
MAPPINGS_PATH = PROJECT_ROOT / "data/01_raw/securities/openfigi_mapping.csv"
MASTER_PATH = PROJECT_ROOT / "data/02_processed/securities/securities_master.csv"

print("Project root:", PROJECT_ROOT)
print("Candidates:", CANDIDATES_PATH)
print("Mappings:", MAPPINGS_PATH)
print("Security Master:", MASTER_PATH)

Project root: C:\Users\MameBARBOSAMONTEIRO\projet01
Candidates: C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_processed\securities\security_candidates.csv
Mappings: C:\Users\MameBARBOSAMONTEIRO\projet01\data\01_raw\securities\openfigi_mapping.csv
Security Master: C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_processed\securities\securities_master.csv


La racine du projet a été correctement identifiée comme :

`C:\Users\MameBARBOSAMONTEIRO\projet01`

Les trois datasets nécessaires à l'analyse du Security Master sont également résolus dans les emplacements attendus :

- **Security Candidates** : `data/02_processed/securities/security_candidates.csv`
- **OpenFIGI Mapping** : `data/01_raw/securities/openfigi_mapping.csv`
- **Security Master** : `data/02_processed/securities/securities_master.csv`

Ce contrôle confirme que le notebook analyse les artefacts produits par le pipeline courant et que les fichiers RAW et préparés restent correctement séparés.

> Contrôle réussi. Aucune anomalie de chemin ou de résolution du projet n'est observée.

## 2. Load the three datasets

All three files are read with strings preserved where practical. This avoids accidentally changing identifiers such as numeric-looking tickers during exploratory analysis.

In [2]:
for path in (CANDIDATES_PATH, MAPPINGS_PATH, MASTER_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

candidates = pd.read_csv(CANDIDATES_PATH, dtype=str, keep_default_na=False)
mappings = pd.read_csv(MAPPINGS_PATH, dtype=str, keep_default_na=False)
master = pd.read_csv(MASTER_PATH, dtype=str, keep_default_na=False)

print("Candidates shape:", candidates.shape)
print("Mappings shape:", mappings.shape)
print("Security Master shape:", master.shape)

Candidates shape: (120, 6)
Mappings shape: (1992, 17)
Security Master shape: (120, 20)


- Candidates: **120**
- OpenFIGI mappings: **1,992**
- Security Master: **120**

Each candidate is represented once in the Security Master.
> Result: PASS

## 3. Structural checks

These checks confirm that the analysis operates on the expected lineage:

- one Security Master row per Security Candidate;
- OpenFIGI `input_position` values point into Security Candidates;
- OpenFIGI `id_value` agrees with the candidate ticker at that position.

In [3]:
required_candidate_columns = {
    "Ticker", "Name", "Location", "Exchange", "Currency", "Asset Class"
}
required_mapping_columns = {
    "input_position", "id_value", "result_position", "figi", "ticker", "name",
    "exch_code", "market_sector", "security_type", "security_type_2",
    "security_description", "composite_figi", "share_class_figi",
    "provider_warning", "provider_error"
}
required_master_columns = {
    "Ticker", "Exchange", "figi", "resolution_status", "resolution_rule",
    "mapping_candidates", "expected_exch_code"
}

assert required_candidate_columns.issubset(candidates.columns)
assert required_mapping_columns.issubset(mappings.columns)
assert required_master_columns.issubset(master.columns)
assert len(master) == len(candidates), "Master/candidate row-count mismatch"

mappings_audit = mappings.copy()
mappings_audit["input_position_int"] = pd.to_numeric(
    mappings_audit["input_position"], errors="raise"
).astype(int)

assert mappings_audit["input_position_int"].between(0, len(candidates) - 1).all()

candidate_ticker_by_position = candidates.reset_index(drop=True)["Ticker"]
lineage = mappings_audit[["input_position_int", "id_value"]].copy()
lineage["candidate_ticker"] = lineage["input_position_int"].map(candidate_ticker_by_position)
lineage["ticker_matches"] = (
    lineage["id_value"].str.strip().str.casefold()
    == lineage["candidate_ticker"].str.strip().str.casefold()
)

print("Master rows equal candidates:", len(master) == len(candidates))
print("Mapping lineage matches:", lineage["ticker_matches"].all())
print("Lineage mismatches:", int((~lineage["ticker_matches"]).sum()))

Master rows equal candidates: True
Mapping lineage matches: True
Lineage mismatches: 0


### Lineage validation

Master-to-candidate alignment is complete, with **0 mapping lineage mismatches**.
> Result: PASS

## 4. Security Master resolution summary

This is the top-level diagnostic view. The goal is not to maximize `RESOLVED` mechanically. The goal is to understand why each unresolved or ambiguous security remains unresolved.

In [4]:
status_summary = (
    master["resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("resolution_status")
    .reset_index(name="records")
)
status_summary["percentage"] = (status_summary["records"] / len(master) * 100).round(2)
status_summary

,resolution_status,records,percentage
0,RESOLVED,100,83.33
1,AMBIGUOUS,11,9.17
2,UNRESOLVED,9,7.50


> The majority of candidates are resolved, while **20 cases require further review**.

In [5]:
rule_summary = (
    master.groupby(["resolution_status", "resolution_rule"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values(["resolution_status", "records"], ascending=[True, False])
    .reset_index(drop=True)
)
rule_summary

,resolution_status,resolution_rule,records
0,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,11
1,RESOLVED,EXCHANGE_MATCH,74
2,RESOLVED,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH,25
3,RESOLVED,UNIQUE_FIGI,1
4,UNRESOLVED,UNRESOLVED_NO_FIGI,9


- **99** candidates resolved through exchange-based matching.
- **1** candidate resolved from a unique FIGI.
- **11** remain ambiguous due to multiple FIGIs.
- **9** have no FIGI.

> Exchange matching is the main resolution mechanism. The **20 unresolved or ambiguous cases require further analysis**.


## 5. Non-resolved population

The next view isolates every `AMBIGUOUS` and `UNRESOLVED` security before examining the underlying OpenFIGI rows.

In [6]:
non_resolved = master.loc[
    master["resolution_status"].ne("RESOLVED"),
    [
        "Ticker", "Name", "Location", "Exchange", "Currency", "Asset Class",
        "expected_exch_code", "resolution_status", "resolution_rule", "mapping_candidates"
    ],
].copy()

non_resolved.sort_values(
    ["resolution_status", "Exchange", "Ticker"],
    kind="stable",
).reset_index(drop=True)

,Ticker,Name,Location,Exchange,Currency,Asset Class,expected_exch_code,resolution_status,resolution_rule,mapping_candidates
0,2082,ACWA POWER CO,Saudi Arabia,Saudi Stock Exchange,USD,Equity,AB,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,8
1,000895,HENAN SHUANGHUI INVESTMENT & DEVEL,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
2,001979,CHINA MERCHANTS SHEKOU INDUSTRIAL,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
3,002129,TCL ZHONGHUAN RENEWABLE ENERGY TEC,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
4,002202,GOLDWIND SCIENCE&TECHNOLOGY LTD A,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
5,300274,SUNGROW POWER SUPPLY LTD A,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,4
6,300750,CONTEMPORARY AMPEREX TECHNOLOGY A,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,4
7,301358,HUNAN YUNENG NEW ENERGY BATTERY MA,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
8,1216,UNI-PRESIDENT ENTERPRISES,Taiwan,Taiwan Stock Exchange,USD,Equity,TT,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3
9,6446,PHARMAESSENTIA,Taiwan,Taiwan Stock Exchange,USD,Equity,TT,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,2


### Non-resolved securities

The **20 cases requiring review** are concentrated in specific markets:

- **11 ambiguous:** mainly Shenzhen, Taiwan and Saudi Arabia, where multiple FIGIs remain.
- **9 unresolved:** mainly India, plus Sweden, UK and Mexico, where no FIGI was returned.

> This suggests **market/ticker convention issues rather than a general resolution failure**.

Next (optional): inspect the underlying OpenFIGI candidates before changing any production rule.

## 6. Attach benchmark attributes to every raw OpenFIGI result

`input_position` is the technical lineage key. This produces an analysis table where every OpenFIGI result is shown beside the original benchmark candidate.

In [7]:
benchmark = candidates.reset_index(drop=True).reset_index(names="input_position_int")
benchmark = benchmark.rename(columns={
    "Ticker": "benchmark_ticker",
    "Name": "benchmark_name",
    "Location": "benchmark_location",
    "Exchange": "benchmark_exchange",
    "Currency": "benchmark_currency",
    "Asset Class": "benchmark_asset_class",
})

mapping_detail = mappings_audit.merge(
    benchmark,
    on="input_position_int",
    how="left",
    validate="many_to_one",
)

master_resolution = master[[
    "Ticker", "expected_exch_code", "resolution_status", "resolution_rule"
]].rename(columns={"Ticker": "benchmark_ticker"})

mapping_detail = mapping_detail.merge(
    master_resolution,
    on="benchmark_ticker",
    how="left",
    validate="many_to_one",
)

mapping_detail["ticker_match"] = (
    mapping_detail["ticker"].str.strip().str.casefold()
    == mapping_detail["benchmark_ticker"].str.strip().str.casefold()
)
mapping_detail["exchange_match"] = (
    mapping_detail["exch_code"].str.strip().str.casefold()
    == mapping_detail["expected_exch_code"].str.strip().str.casefold()
)
mapping_detail["equity_match"] = mapping_detail["market_sector"].str.strip().str.casefold().eq("equity")

print("Detailed mapping rows:", len(mapping_detail))

Detailed mapping rows: 1992


All **1,992 OpenFIGI mapping rows** were successfully enriched with benchmark and resolution attributes.

> Result: PASS

## 7. Ambiguous securities and all surviving raw candidates

This table is the main review surface for ambiguity. It deliberately displays all candidate FIGIs instead of selecting one.

In [8]:
ambiguous_tickers = set(
    master.loc[master["resolution_status"].eq("AMBIGUOUS"), "Ticker"]
)

ambiguous_detail = mapping_detail[
    mapping_detail["benchmark_ticker"].isin(ambiguous_tickers)
].copy()

ambiguous_columns = [
    "benchmark_ticker", "benchmark_name", "benchmark_exchange", "benchmark_currency",
    "expected_exch_code", "result_position", "figi", "ticker", "name", "exch_code",
    "market_sector", "security_type", "security_type_2", "security_description",
    "composite_figi", "share_class_figi", "ticker_match", "exchange_match",
    "equity_match", "provider_warning", "provider_error"
]

ambiguous_detail[ambiguous_columns].sort_values(
    ["benchmark_ticker", "result_position"],
    kind="stable",
).reset_index(drop=True)

,benchmark_ticker,benchmark_name,benchmark_exchange,benchmark_currency,expected_exch_code,result_position,figi,ticker,name,exch_code,market_sector,security_type,security_type_2,security_description,composite_figi,share_class_figi,ticker_match,exchange_match,equity_match,provider_warning,provider_error
0,000895,HENAN SHUANGHUI INVESTMENT & DEVEL,Shenzhen Stock Exchange,USD,SZ,0.0,BBG000DYKF22,000895,HENAN SHUANGHUI INVESTMENT-A,CH,Equity,Common Stock,Common Stock,000895,BBG000DYKF22,BBG001SJWFK4,True,False,True,,
1,000895,HENAN SHUANGHUI INVESTMENT & DEVEL,Shenzhen Stock Exchange,USD,SZ,1.0,BBG000DYKFN9,000895,HENAN SHUANGHUI INVESTMENT-A,CS,Equity,Common Stock,Common Stock,000895,BBG000DYKF22,BBG001SJWFK4,True,False,True,,
2,000895,HENAN SHUANGHUI INVESTMENT & DEVEL,Shenzhen Stock Exchange,USD,SZ,2.0,BBG00F136NW6,000895,HENAN SHUANGHUI INVESTMENT-A,C2,Equity,Common Stock,Common Stock,000895,BBG00F136NT0,BBG001SJWFK4,True,False,True,,
3,001979,CHINA MERCHANTS SHEKOU INDUSTRIAL,Shenzhen Stock Exchange,USD,SZ,0.0,BBG000QKV4L0,001979,CHINA MERCHANTS SHEKOU IND-A,CH,Equity,Common Stock,Common Stock,001979,BBG000QKV4L0,BBG001SS9C45,True,False,True,,
4,001979,CHINA MERCHANTS SHEKOU INDUSTRIAL,Shenzhen Stock Exchange,USD,SZ,1.0,BBG00BS2FZ24,001979,CHINA MERCHANTS SHEKOU IND-A,CS,Equity,Common Stock,Common Stock,001979,BBG000QKV4L0,BBG001SS9C45,True,False,True,,
5,001979,CHINA MERCHANTS SHEKOU INDUSTRIAL,Shenzhen Stock Exchange,USD,SZ,2.0,BBG00DY66236,001979,CHINA MERCHANTS SHEKOU IND-A,C2,Equity,Common Stock,Common Stock,001979,BBG00DY66227,BBG001SS9C45,True,False,True,,
6,002129,TCL ZHONGHUAN RENEWABLE ENERGY TEC,Shenzhen Stock Exchange,USD,SZ,0.0,BBG000PK1KS1,002129,TCL ZHONGHUAN RENEWABLE EN-A,CH,Equity,Common Stock,Common Stock,002129,BBG000PK1KS1,BBG001SR2ZK5,True,False,True,,
7,002129,TCL ZHONGHUAN RENEWABLE ENERGY TEC,Shenzhen Stock Exchange,USD,SZ,1.0,BBG000PK1M52,002129,TCL ZHONGHUAN RENEWABLE EN-A,CS,Equity,Common Stock,Common Stock,002129,BBG000PK1KS1,BBG001SR2ZK5,True,False,True,,
8,002129,TCL ZHONGHUAN RENEWABLE ENERGY TEC,Shenzhen Stock Exchange,USD,SZ,2.0,BBG00F137536,002129,TCL ZHONGHUAN RENEWABLE EN-A,C2,Equity,Common Stock,Common Stock,002129,BBG00F137527,BBG001SR2ZK5,True,False,True,,
9,002202,GOLDWIND SCIENCE&TECHNOLOGY LTD A,Shenzhen Stock Exchange,USD,SZ,0.0,BBG000BN94X2,002202,GOLDWIND SCIENCE & TECHNOL-A,CH,Equity,Common Stock,Common Stock,002202,BBG000BN94X2,BBG001SF32L2,True,False,True,,


The ambiguous cases show a clear pattern:

- **Shenzhen:** benchmark tickers match, but OpenFIGI returns multiple exchange variants (`CH`, `CS`, `C2`, `ER`) rather than the expected `SZ`.
- **Taiwan:** the primary-looking results use `TT (Taiwan Stock Exchange)`, so the current exact comparison with `TT` fails.
- **Saudi Arabia:** ticker `2082` maps to unrelated ETP/Mutual Fund instruments, so no automatic selection is justified.
- Most Shenzhen alternatives share the same **share-class FIGI**, suggesting multiple listings of the same share class.

> The ambiguity is largely caused by exchange-code representation and multiple listings. No new selection rule should be applied yet.

## 8. Ambiguity diagnostics by ticker

The aggregation below counts distinct FIGIs, composite FIGIs, share-class FIGIs, exchange codes, security types, and returned tickers. It helps identify whether several rows may represent the same underlying share class or genuinely different instruments.

In [9]:
def nunique_nonblank(series):
    values = series.astype(str).str.strip()
    return values[values.ne("")].nunique()

ambiguous_diagnostics = (
    ambiguous_detail
    .groupby(
        ["benchmark_ticker", "benchmark_exchange", "expected_exch_code"],
        dropna=False,
    )
    .agg(
        raw_rows=("figi", "size"),
        distinct_figis=("figi", nunique_nonblank),
        distinct_composite_figis=("composite_figi", nunique_nonblank),
        distinct_share_class_figis=("share_class_figi", nunique_nonblank),
        distinct_returned_tickers=("ticker", nunique_nonblank),
        distinct_exchange_codes=("exch_code", nunique_nonblank),
        distinct_security_types=("security_type", nunique_nonblank),
        distinct_security_type_2=("security_type_2", nunique_nonblank),
        exact_ticker_matches=("ticker_match", "sum"),
        exchange_matches=("exchange_match", "sum"),
        equity_matches=("equity_match", "sum"),
    )
    .reset_index()
)

ambiguous_diagnostics

,benchmark_ticker,benchmark_exchange,expected_exch_code,raw_rows,distinct_figis,distinct_composite_figis,distinct_share_class_figis,distinct_returned_tickers,distinct_exchange_codes,distinct_security_types,distinct_security_type_2,exact_ticker_matches,exchange_matches,equity_matches
0,000895,Shenzhen Stock Exchange,SZ,3,3,2,1,1,3,1,1,3,0,3
1,001979,Shenzhen Stock Exchange,SZ,3,3,2,1,1,3,1,1,3,0,3
2,002129,Shenzhen Stock Exchange,SZ,3,3,2,1,1,3,1,1,3,0,3
3,002202,Shenzhen Stock Exchange,SZ,3,3,2,1,1,3,1,1,3,0,3
4,1216,Taiwan Stock Exchange,TT,3,3,3,2,1,3,1,1,3,0,3
5,2082,Saudi Stock Exchange,AB,8,8,2,1,1,8,1,1,8,0,8
6,300274,Shenzhen Stock Exchange,SZ,4,4,3,1,1,4,1,1,4,0,4
7,300750,Shenzhen Stock Exchange,SZ,4,4,3,1,1,4,1,1,4,0,4
8,301358,Shenzhen Stock Exchange,SZ,3,3,2,1,1,3,1,1,3,0,3
9,6446,Taiwan Stock Exchange,TT,2,2,2,1,1,2,1,1,2,0,2


All **11 ambiguous securities** have exact ticker and Equity matches, but **none has an exact exchange-code match**.

Most cases also share a single `share_class_figi`, indicating multiple listings of the same share class rather than different securities.

> Exchange-code normalization is the main issue. Share-class relationships should be investigated before adding any new resolution rule.

## 9. Compact candidate lists for each ambiguous security

This representation makes manual review easier while keeping all relevant OpenFIGI attributes visible.

In [10]:
def compact_candidates(group):
    records = []
    for _, row in group.iterrows():
        records.append({
            "figi": row["figi"],
            "ticker": row["ticker"],
            "name": row["name"],
            "exch_code": row["exch_code"],
            "market_sector": row["market_sector"],
            "security_type": row["security_type"],
            "security_type_2": row["security_type_2"],
            "composite_figi": row["composite_figi"],
            "share_class_figi": row["share_class_figi"],
            "ticker_match": bool(row["ticker_match"]),
            "exchange_match": bool(row["exchange_match"]),
            "equity_match": bool(row["equity_match"]),
        })
    return records

ambiguous_candidate_lists = {
    ticker: compact_candidates(group)
    for ticker, group in ambiguous_detail.groupby("benchmark_ticker", sort=False)
}

for ticker, records in ambiguous_candidate_lists.items():
    benchmark_row = master.loc[master["Ticker"].eq(ticker)].iloc[0]
    print("=" * 100)
    print(
        f"{ticker} | {benchmark_row['Name']} | "
        f"{benchmark_row['Exchange']} | expected={benchmark_row['expected_exch_code']}"
    )
    print(json.dumps(records, indent=2, ensure_ascii=False))

6669 | WIWYNN CORPORATION | Taiwan Stock Exchange | expected=TT
[
  {
    "figi": "BBG00J4ZF054",
    "ticker": "6669",
    "name": "WIWYNN CORP",
    "exch_code": "TT (Taiwan Stock Exchange)",
    "market_sector": "Equity",
    "security_type": "Common Stock",
    "security_type_2": "Common Stock",
    "composite_figi": "BBG00J4ZF036",
    "share_class_figi": "BBG00J4ZF090",
    "ticker_match": true,
    "exchange_match": false,
    "equity_match": true
  },
  {
    "figi": "BBG0123Q7KQ3",
    "ticker": "6669",
    "name": "ACOTEC SCIENTIFIC HOLDINGS L",
    "exch_code": "HK",
    "market_sector": "Equity",
    "security_type": "Common Stock",
    "security_type_2": "Common Stock",
    "composite_figi": "BBG00ZQYS2M0",
    "share_class_figi": "BBG00ZQYS2P7",
    "ticker_match": true,
    "exchange_match": false,
    "equity_match": true
  },
  {
    "figi": "BBG0250812Q4",
    "ticker": "6669",
    "name": "WIWYNN CORP",
    "exch_code": "ER",
    "market_sector": "Equity",
    "secur

The detailed review confirms that ticker matching alone is unsafe: ticker `2082` matches unrelated securities.

> **Decision:** retain ambiguous status when exchange and instrument identity cannot uniquely identify the benchmark security.

## 10. Unresolved securities

For unresolved securities, distinguish between:

- mappings present but without a usable FIGI;
- provider warnings/errors;
- no useful result despite the ticker submission.

In [11]:
unresolved_tickers = set(
    master.loc[master["resolution_status"].eq("UNRESOLVED"), "Ticker"]
)

unresolved_detail = mapping_detail[
    mapping_detail["benchmark_ticker"].isin(unresolved_tickers)
].copy()

unresolved_detail[[
    "benchmark_ticker", "benchmark_name", "benchmark_exchange", "benchmark_currency",
    "expected_exch_code", "id_value", "result_position", "figi", "ticker", "name",
    "exch_code", "market_sector", "security_type", "security_type_2",
    "provider_warning", "provider_error"
]].sort_values(
    ["benchmark_ticker", "result_position"],
    kind="stable",
).reset_index(drop=True)

,benchmark_ticker,benchmark_name,benchmark_exchange,benchmark_currency,expected_exch_code,id_value,result_position,figi,ticker,name,exch_code,market_sector,security_type,security_type_2,provider_warning,provider_error
0,COLPAL,COLGATE PALMOLIVE INDIA LTD,National Stock Exchange Of India,USD,IN,COLPAL,,,,,,,,,No identifier found.,
1,ESSITY B,ESSITY CLASS B,Nasdaq Omx Nordic,USD,SS,ESSITY B,,,,,,,,,No identifier found.,
2,FIBRAPL14,PROLOGIS PROPERTY MEXICO REIT,Bolsa Mexicana De Valores,USD,MM,FIBRAPL14,,,,,,,,,No identifier found.,
3,HINDUNILVR,HINDUSTAN UNILEVER LTD,National Stock Exchange Of India,USD,IN,HINDUNILVR,,,,,,,,,No identifier found.,
4,LUPIN,LUPIN,National Stock Exchange Of India,USD,IN,LUPIN,,,,,,,,,No identifier found.,
5,NESTLEIND,NESTLE INDIA LTD,National Stock Exchange Of India,USD,IN,NESTLEIND,,,,,,,,,No identifier found.,
6,SCA B,SVENSKA CELLULOSA B,Nasdaq Omx Nordic,USD,SS,SCA B,,,,,,,,,No identifier found.,
7,SUZLON,SUZLON ENERGY,National Stock Exchange Of India,USD,IN,SUZLON,,,,,,,,,No identifier found.,
8,UU.,UNITED UTILITIES GROUP PLC,London Stock Exchange,USD,LN,UU.,,,,,,,,,No identifier found.,


All **9 unresolved securities** received the OpenFIGI warning `No identifier found.`

> These cases require ticker-format or identifier investigation. No change to the Security Master resolution rules is justified yet.

In [12]:
unresolved_summary = (
    unresolved_detail
    .groupby(
        ["benchmark_ticker", "benchmark_exchange", "expected_exch_code"],
        dropna=False,
    )
    .agg(
        raw_rows=("id_value", "size"),
        nonblank_figis=("figi", nunique_nonblank),
        warnings=("provider_warning", nunique_nonblank),
        errors=("provider_error", nunique_nonblank),
    )
    .reset_index()
)

unresolved_summary

,benchmark_ticker,benchmark_exchange,expected_exch_code,raw_rows,nonblank_figis,warnings,errors
0,COLPAL,National Stock Exchange Of India,IN,1,0,1,0
1,ESSITY B,Nasdaq Omx Nordic,SS,1,0,1,0
2,FIBRAPL14,Bolsa Mexicana De Valores,MM,1,0,1,0
3,HINDUNILVR,National Stock Exchange Of India,IN,1,0,1,0
4,LUPIN,National Stock Exchange Of India,IN,1,0,1,0
5,NESTLEIND,National Stock Exchange Of India,IN,1,0,1,0
6,SCA B,Nasdaq Omx Nordic,SS,1,0,1,0
7,SUZLON,National Stock Exchange Of India,IN,1,0,1,0
8,UU.,London Stock Exchange,LN,1,0,1,0


Each unresolved ticker produced **one OpenFIGI response, zero FIGIs, one warning and no errors**.

> This confirms a systematic identifier-matching issue rather than an API failure.

## 11. Inspect warnings and provider errors

This cell exposes the distinct provider messages associated with unresolved cases without modifying the raw mapping dataset.

In [13]:
provider_messages = unresolved_detail.loc[
    unresolved_detail["provider_warning"].ne("")
    | unresolved_detail["provider_error"].ne(""),
    ["benchmark_ticker", "id_value", "provider_warning", "provider_error"],
].drop_duplicates().reset_index(drop=True)

provider_messages

,benchmark_ticker,id_value,provider_warning,provider_error
0,ESSITY B,ESSITY B,No identifier found.,
1,UU.,UU.,No identifier found.,
2,SUZLON,SUZLON,No identifier found.,
3,SCA B,SCA B,No identifier found.,
4,HINDUNILVR,HINDUNILVR,No identifier found.,
5,LUPIN,LUPIN,No identifier found.,
6,NESTLEIND,NESTLEIND,No identifier found.,
7,FIBRAPL14,FIBRAPL14,No identifier found.,
8,COLPAL,COLPAL,No identifier found.,


All 9 unresolved cases consistently return `No identifier found.` with the submitted ticker unchanged.

> The unresolved status is correctly classified; further investigation should focus on alternative ticker conventions or identifiers.

## 12. Audit resolved securities

Resolution quality needs validation too. This section samples each production resolution rule and checks whether the selected FIGI can be found in the raw mappings for the corresponding candidate.

In [14]:
resolved = master[master["resolution_status"].eq("RESOLVED")].copy()

resolved_audit_rows = []
for _, row in resolved.iterrows():
    ticker = row["Ticker"]
    figi = row["figi"]
    source_rows = mapping_detail[
        mapping_detail["benchmark_ticker"].eq(ticker)
        & mapping_detail["figi"].eq(figi)
    ]
    resolved_audit_rows.append({
        "Ticker": ticker,
        "resolution_rule": row["resolution_rule"],
        "figi": figi,
        "raw_mapping_found": not source_rows.empty,
        "raw_rows_for_selected_figi": len(source_rows),
    })

resolved_audit = pd.DataFrame(resolved_audit_rows)
print("All selected FIGIs found in raw mapping:", resolved_audit["raw_mapping_found"].all())
resolved_audit.groupby(["resolution_rule", "raw_mapping_found"]).size().reset_index(name="records")

All selected FIGIs found in raw mapping: True


,resolution_rule,raw_mapping_found,records
0,EXCHANGE_MATCH,True,74
1,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH,True,25
2,UNIQUE_FIGI,True,1


All **100 selected FIGIs** are present in the raw OpenFIGI mapping dataset across every resolution rule.

> Result: PASS. Selected identifiers maintain complete source lineage.

In [15]:
# Deterministic sample: first five resolved securities per resolution rule.
resolved_samples = (
    resolved.sort_values(["resolution_rule", "Ticker"], kind="stable")
    .groupby("resolution_rule", group_keys=False)
    .head(5)
)

resolved_samples[[
    "Ticker", "Name", "Exchange", "expected_exch_code", "figi",
    "openfigi_ticker", "openfigi_name", "exch_code", "market_sector",
    "security_type", "security_type_2", "resolution_rule"
]].reset_index(drop=True)

,Ticker,Name,Exchange,expected_exch_code,figi,openfigi_ticker,openfigi_name,exch_code,market_sector,security_type,security_type_2,resolution_rule
0,010120,LS ELECTRIC LTD,Korea Exchange (Stock Market),KS,BBG000GPVN92,010120,LS ELECTRIC CO LTD,KS,Equity,Common Stock,Common Stock,EXCHANGE_MATCH
1,1044,HENGAN INTERNATIONAL GROUP,Hong Kong Exchanges And Clearing Ltd,HK,BBG000CJXNB5,1044,HENGAN INTL GROUP CO LTD,HK,Equity,Common Stock,Common Stock,EXCHANGE_MATCH
2,1093,CSPC PHARMACEUTICAL GROUP,Hong Kong Exchanges And Clearing Ltd,HK,BBG000C3N3B5,1093,CSPC PHARMACEUTICAL GROUP LT,HK,Equity,Common Stock,Common Stock,EXCHANGE_MATCH
3,1585,YADEA GROUP HOLDINGS,Hong Kong Exchanges And Clearing Ltd,HK,BBG00CW0KK19,1585,YADEA GROUP HOLDINGS LTD,HK,Equity,Common Stock,Common Stock,EXCHANGE_MATCH
4,1801,INNOVENT BIOLOGICS,Hong Kong Exchanges And Clearing Ltd,HK,BBG00M8832B5,1801,INNOVENT BIOLOGICS INC,HK,Equity,Common Stock,Common Stock,EXCHANGE_MATCH
5,12,HENDERSON LAND DEVELOPMENT LTD,Hong Kong Exchanges And Clearing Ltd,HK,BBG000BDWF36,12,HENDERSON LAND DEVELOPMENT,HK,Equity,Common Stock,Common Stock,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH
6,2015,LI AUTO CLASS A,Hong Kong Exchanges And Clearing Ltd,HK,BBG0120WC125,2015,LI AUTO INC-CLASS A,HK,Equity,Common Stock,Common Stock,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH
7,BMY,BRISTOL MYERS SQUIBB,NYSE,US,BBG000DQLV23,BMY,BRISTOL-MYERS SQUIBB CO,US,Equity,Common Stock,Common Stock,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH
8,CLAR,CAPITALAND ASCENDAS REIT,Singapore Exchange,SP,BBG000PFLWB2,CLAR,CAPITALAND ASCENDAS REIT,SP,Equity,REIT,REIT,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH
9,COV,COVIVIO SA,Nyse Euronext - Euronext Paris,FP,BBG000BJ2HD1,COV,COVIVIO,FP,Equity,REIT,REIT,MARKET_SECTOR_EQUITY_EXCHANGE_MATCH


The sampled resolved securities show consistent benchmark-to-OpenFIGI alignment across ticker, exchange and instrument type, including Common Stocks and REITs.

> Result: PASS. No obvious false-positive resolution is observed in the sample.

## 13. Potential additional discriminators

The next cells do **not** create a new production rule. They only measure whether individual attributes might reduce ambiguity.

A new production rule should be considered only if the rule has a clear semantic justification and does not merely improve the resolution percentage.

In [16]:
ambiguity_discriminator_summary = []

for ticker, group in ambiguous_detail.groupby("benchmark_ticker", sort=False):
    usable = group[
        group["figi"].ne("")
        & group["provider_error"].eq("")
    ].copy()

    figis = usable["figi"].nunique()
    exact_ticker_figis = usable.loc[usable["ticker_match"], "figi"].nunique()
    exchange_figis = usable.loc[usable["exchange_match"], "figi"].nunique()
    equity_figis = usable.loc[usable["equity_match"], "figi"].nunique()

    common_stock_mask = (
        usable["security_type_2"].str.strip().str.casefold().eq("common stock")
        | usable["security_type"].str.strip().str.casefold().eq("common stock")
    )
    common_stock_figis = usable.loc[common_stock_mask, "figi"].nunique()

    ambiguity_discriminator_summary.append({
        "Ticker": ticker,
        "current_distinct_figis": figis,
        "figis_after_exact_ticker_only": exact_ticker_figis,
        "figis_after_exchange_only": exchange_figis,
        "figis_after_equity_only": equity_figis,
        "figis_after_common_stock_only": common_stock_figis,
    })

pd.DataFrame(ambiguity_discriminator_summary)

,Ticker,current_distinct_figis,figis_after_exact_ticker_only,figis_after_exchange_only,figis_after_equity_only,figis_after_common_stock_only
0,6669,3,3,0,3,3
1,1216,3,3,0,3,3
2,300750,4,4,0,4,4
3,002129,3,3,0,3,3
4,001979,3,3,0,3,3
5,300274,4,4,0,4,4
6,002202,3,3,0,3,3
7,301358,3,3,0,3,3
8,2082,8,8,0,8,0
9,000895,3,3,0,3,3


Ticker, Equity classification and Common Stock type **do not reduce the ambiguity** for the affected securities.

The current exchange test returns zero matches, confirming that the next investigation should focus on **exchange-code normalization**, not additional ticker or security-type filtering.

## 14. Share-class and composite-FIGI structure

Several distinct FIGIs can belong to the same share class or composite security. This cell exposes those relationships for ambiguous cases. The presence of a common `share_class_figi` or `composite_figi` is evidence to investigate, not by itself permission to choose one listing FIGI.

In [17]:
ambiguous_relationships = ambiguous_detail.loc[
    ambiguous_detail["figi"].ne(""),
    [
        "benchmark_ticker", "figi", "ticker", "exch_code", "security_type",
        "security_type_2", "composite_figi", "share_class_figi",
        "ticker_match", "exchange_match", "equity_match"
    ],
].drop_duplicates()

ambiguous_relationships.sort_values(
    ["benchmark_ticker", "share_class_figi", "composite_figi", "figi"],
    kind="stable",
).reset_index(drop=True)

,benchmark_ticker,figi,ticker,exch_code,security_type,security_type_2,composite_figi,share_class_figi,ticker_match,exchange_match,equity_match
0,000895,BBG000DYKF22,000895,CH,Common Stock,Common Stock,BBG000DYKF22,BBG001SJWFK4,True,False,True
1,000895,BBG000DYKFN9,000895,CS,Common Stock,Common Stock,BBG000DYKF22,BBG001SJWFK4,True,False,True
2,000895,BBG00F136NW6,000895,C2,Common Stock,Common Stock,BBG00F136NT0,BBG001SJWFK4,True,False,True
3,001979,BBG000QKV4L0,001979,CH,Common Stock,Common Stock,BBG000QKV4L0,BBG001SS9C45,True,False,True
4,001979,BBG00BS2FZ24,001979,CS,Common Stock,Common Stock,BBG000QKV4L0,BBG001SS9C45,True,False,True
5,001979,BBG00DY66236,001979,C2,Common Stock,Common Stock,BBG00DY66227,BBG001SS9C45,True,False,True
6,002129,BBG000PK1KS1,002129,CH,Common Stock,Common Stock,BBG000PK1KS1,BBG001SR2ZK5,True,False,True
7,002129,BBG000PK1M52,002129,CS,Common Stock,Common Stock,BBG000PK1KS1,BBG001SR2ZK5,True,False,True
8,002129,BBG00F137536,002129,C2,Common Stock,Common Stock,BBG00F137527,BBG001SR2ZK5,True,False,True
9,002202,BBG000BN94X2,002202,CH,Common Stock,Common Stock,BBG000BN94X2,BBG001SF32L2,True,False,True


Most ambiguous China and Taiwan mappings share the same `share_class_figi`, confirming that several FIGIs represent alternative listings of the same share class.

Ticker and security type therefore cannot determine the correct listing FIGI alone.

**Conclusion:** `share_class_figi` is useful for confirming security identity, but exchange/listing identification remains necessary before selecting a FIGI.

## 15. Exchange mapping coverage

This summarizes where unresolved or ambiguous cases are concentrated. A valid `expected_exch_code` does not guarantee uniqueness, but missing exchange coverage is immediately visible here.

In [18]:
exchange_resolution_summary = (
    master.groupby(
        ["Exchange", "expected_exch_code", "resolution_status"],
        dropna=False,
    )
    .size()
    .reset_index(name="records")
    .sort_values(["Exchange", "resolution_status"], kind="stable")
    .reset_index(drop=True)
)

exchange_resolution_summary

,Exchange,expected_exch_code,resolution_status,records
0,Asx - All Markets,AU,RESOLVED,3
1,Bolsa De Madrid,SM,RESOLVED,1
2,Bolsa Mexicana De Valores,MM,RESOLVED,2
3,Bolsa Mexicana De Valores,MM,UNRESOLVED,1
4,Borsa Italiana,IM,RESOLVED,1
5,Hong Kong Exchanges And Clearing Ltd,HK,RESOLVED,21
6,Johannesburg Stock Exchange,SJ,RESOLVED,1
7,Korea Exchange (Kosdaq),KQ,RESOLVED,1
8,Korea Exchange (Stock Market),KS,RESOLVED,2
9,London Stock Exchange,LN,RESOLVED,3


Most exchanges resolve successfully. Remaining issues are concentrated in **Shenzhen, Taiwan, India and Saudi Arabia**, with isolated cases in Mexico, London and Sweden.

**Conclusion:** resolution failures are market-specific rather than systemic.

## 16. Investigation worksheet

The table below is intended for human review. Add observations in a separate copy or downstream documentation rather than modifying the production CSV files.

In [19]:
review = non_resolved.copy()
review["manual_observation"] = ""
review["candidate_rule"] = ""
review["evidence_sufficient"] = ""
review["recommended_action"] = ""

review.reset_index(drop=True)

,Ticker,Name,Location,Exchange,Currency,Asset Class,expected_exch_code,resolution_status,resolution_rule,mapping_candidates,manual_observation,candidate_rule,evidence_sufficient,recommended_action
0,6669,WIWYNN CORPORATION,Taiwan,Taiwan Stock Exchange,USD,Equity,TT,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3,,,,
1,1216,UNI-PRESIDENT ENTERPRISES,Taiwan,Taiwan Stock Exchange,USD,Equity,TT,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3,,,,
2,ESSITY B,ESSITY CLASS B,Sweden,Nasdaq Omx Nordic,USD,Equity,SS,UNRESOLVED,UNRESOLVED_NO_FIGI,0,,,,
3,300750,CONTEMPORARY AMPEREX TECHNOLOGY A,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,4,,,,
4,UU.,UNITED UTILITIES GROUP PLC,United Kingdom,London Stock Exchange,USD,Equity,LN,UNRESOLVED,UNRESOLVED_NO_FIGI,0,,,,
5,002129,TCL ZHONGHUAN RENEWABLE ENERGY TEC,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3,,,,
6,SUZLON,SUZLON ENERGY,India,National Stock Exchange Of India,USD,Equity,IN,UNRESOLVED,UNRESOLVED_NO_FIGI,0,,,,
7,SCA B,SVENSKA CELLULOSA B,Sweden,Nasdaq Omx Nordic,USD,Equity,SS,UNRESOLVED,UNRESOLVED_NO_FIGI,0,,,,
8,HINDUNILVR,HINDUSTAN UNILEVER LTD,India,National Stock Exchange Of India,USD,Equity,IN,UNRESOLVED,UNRESOLVED_NO_FIGI,0,,,,
9,001979,CHINA MERCHANTS SHEKOU INDUSTRIAL,China,Shenzhen Stock Exchange,USD,Equity,SZ,AMBIGUOUS,AMBIGUOUS_MULTIPLE_FIGIS,3,,,,


The review table captures all **20 non-resolved securities** and provides dedicated fields for documenting evidence and recommended actions.

**Decision:** retain these cases for later investigation and proceed with the 100 currently resolved securities.

## 17. Export diagnostic tables only

The following cell is intentionally disabled by default. If enabled, it writes **diagnostic** CSV files under `notebooks/` and never overwrites production datasets.

In [20]:
EXPORT_DIAGNOSTICS = False

if EXPORT_DIAGNOSTICS:
    diagnostics_dir = PROJECT_ROOT / "notebooks"
    diagnostics_dir.mkdir(parents=True, exist_ok=True)

    ambiguous_detail.to_csv(
        diagnostics_dir / "security_master_ambiguous_details.csv",
        index=False,
        encoding="utf-8",
    )
    unresolved_detail.to_csv(
        diagnostics_dir / "security_master_unresolved_details.csv",
        index=False,
        encoding="utf-8",
    )
    review.to_csv(
        diagnostics_dir / "security_master_resolution_review.csv",
        index=False,
        encoding="utf-8",
    )

    print("Diagnostic exports written to:", diagnostics_dir)
else:
    print("Diagnostic export disabled. Set EXPORT_DIAGNOSTICS=True to enable.")

Diagnostic export disabled. Set EXPORT_DIAGNOSTICS=True to enable.


## 18. Conclusion

### Resolution quality

- **120** Security Candidates analyzed
- **100 RESOLVED (83.33%)**
- **11 AMBIGUOUS (9.17%)**
- **9 UNRESOLVED (7.50%)**

### Findings

- All resolved FIGIs maintain complete lineage to the raw OpenFIGI mappings.
- Ambiguities are concentrated in specific markets and mainly concern multiple listing representations.
- Ticker, Equity classification and security type do not safely resolve the remaining ambiguities.
- Unresolved cases consistently received `No identifier found.` from OpenFIGI.

### Decision

The current **83.33% resolution rate is sufficient to continue the portfolio pipeline using only RESOLVED securities**.

AMBIGUOUS and UNRESOLVED securities remain excluded from downstream processing until a later dedicated investigation.

**No additional resolution rule is introduced at this stage.**